# readme
This notebook is for training U-Net and variants for the cardiac MRI segmentation task. 

Most of the work is being done on the GPU PC in the TIC lab, but due to limited GPU power, I can't run experiments with semi- to large- batch sizes. GPUs here are bigger, so I can do train/val experiments with batches > 16 (64 and 128 probs).

# Drive access

In [1]:
# Run this block and follow the URL instructions.
from google.colab import drive # import Gdrive
drive.mount('/content/gdrive') # mount drive to notebook

Mounted at /content/gdrive


# settings.py
training params, model params, directories, etc

In [2]:
import os

##########################################################################
#                            Training settings                           #
##########################################################################

# Hyperparameters
momentum=0.9
weight_decay=0
milestones= [50,100]
gamma=0.1
beta1=0.9
beta2=0.999
lr=1e-3
lr_scheduler='cosineWR'
T_0=50
T_mult=1
T_up=10
optimizer='adam'
step=20
num_epochs=200
batch_size=200
num_workers=4
input_resize=128
in_channels=3
n_classes=4

# Data Directories
root_path = 'gdrive/My Drive/CardiacMRISeg'
root_trainData = os.path.join(root_path,'training/*')
root_testData = os.path.join(root_path,'test/*')

# Output directories
weights_path = os.path.join(root_path, 'experiments/exp2/model_checkpoints')  
os.makedirs(weights_path, exist_ok=True)
stats_root = os.path.join(root_path, 'experiments/exp2/logs')
os.makedirs(stats_root, exist_ok=True)

# Others
val_percent=0.1
log_every=20
save_model=False
data_aug=False
transform=True
model_type='U-Net'




# dataset.py
dataloader 

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Thu Aug 29 10:02:28 2019

@author: hesun

Adapted by calum. 

Updates:
    18.11.20:
        - added function for resizing and normalising data: _transformImage()

"""
from __future__ import print_function
import numpy as np
import torch
import torch.utils.data as data
import torchvision.transforms as tfs
import glob
import cv2
from PIL import Image
from skimage import filters

class dataset(data.Dataset):
    def __init__(self,root, train=True, transform=False):
        self.train = train
        self.transform = transform
#        self.file_list = glob.glob(root) # for running all training images through (He)
        if self.train:
            self.file_list = root
        else:
            self.file_list = glob.glob(root)
            
    def _transformImage(self,img):
        transform = tfs.Compose([
                  tfs.ToTensor(), 
#                  tfs.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]) 
        return transform(img)

    def __getitem__(self,index):
        if self.train:
            file_index = self.file_list[index]
            combine = np.load(file_index)
            # Images
            data = combine[0,:,:]
            data = Image.fromarray(np.int16(data))
#            data = np.array(data)
#            data = data[np.newaxis,:,:]
            # Labels
            label = combine[1,:,:]
            label = Image.fromarray(np.uint8(label))
#            label = np.array(label)
            # Edges
#            edge = filters.sobel(label)
#            edge[edge!=0] = 1
#            edge = np.int8(edge)
            # Option to resize and normalise data
            if self.transform:
                data = data.resize((input_resize, input_resize))
                data = _convertToRGB(data).astype(np.float32)
                data = self._transformImage(data)
                label = label.resize((input_resize, input_resize))
#                edge = edge.resize((input_resize, input_resize))
                return data, torch.from_numpy(np.array(label)).float()#, torch.from_numpy(edge).float()
            else:    
                return torch.from_numpy(np.array(data)).float(), torch.from_numpy(np.array(label)).float()#, torch.from_numpy(edge).float()
        else:
            file_index = self.file_list[index]
            combine = np.load(file_index)
            data = combine[0,:,:]
            data = Image.fromarray(np.int16(data))
            #data = data.resize((128,128))
            data = np.array(data)
            data = data[np.newaxis,:,:]
            label = combine[1,:,:]
            label = Image.fromarray(np.uint8(label))
            #label = label.resize((128,128))
            label = np.array(label)
            if self.transform:
                data = _convertToRGB(data)
                data = self._transformImage(data)
                label = Resize(label)
                return data, label
            else:
                return torch.from_numpy(data).float(), torch.from_numpy(label).float()
    def __len__(self):
        return len(self.file_list)
    
""" Stack pixels over RGB channels """
def _convertToRGB(image):
#    image = np.transpose(image, (1,2,0)) #reshape from (1,H,W) to (H,W,1)
    return np.stack((image, image, image), axis=2)  # new shape: (H, W, 3)
         


# loss.py
contains code for making our loss functions (Dice, focal, ranking)

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable

class DiceLoss(nn.Module):
    def __init__(self, class_num=4,smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
        self.class_num = class_num

    def forward(self,input, target):
        #input = F.log_softmax(input, dim=1)
        input = torch.exp(input)
        self.smooth = 0.1
        Dice = Variable(torch.Tensor([0]).float(), requires_grad=True).cuda()
        for i in range(1,self.class_num):
            input_i = input[:,i,:,:]
            target_i = (target == i).float()
            intersect = (input_i*target_i).sum()
            union = torch.sum(input_i) + torch.sum(target_i)
            if target_i.sum() == 0:
                dice = Variable(torch.Tensor([1]).float()).cuda()
            else:
                dice = (2 * intersect + self.smooth) / (union + self.smooth)
            Dice += dice
        dice_loss = 1 - Dice/(self.class_num - 1)
        return dice_loss
    
"""
======================
Ranking loss
======================
"""
class RankingLoss(nn.Module):
    def __init__(self, class_num=4):
        super(RankingLoss, self).__init__()
        self.class_num = class_num
        self.gamma = 2
        self
    def forward(self, output, target, dist_map, class_map=None):
        ce_loss = Variable(torch.Tensor([0]).float()).cuda()
        i = 0
#        ce_list = []
        for i in range(1, self.class_num):   
            output_i = output[:,i,:,:]     # extract prediction map for ith class (log.softmax format)
            target_i = target[:,i,:,:]     # extract the OHE target for ith class 
            weightmap = dist_map  
            ce = -torch.sum(weightmap * target_i * torch.log(output_i), dim=0)  # compute CE for ith class, where CE = target * log(p), where output_i = softmax predictions
            ce = ce / 262144    # Mean loss across all pixels for the ith class: this reduces the loss by the number of samples (ie ce/262144)
            ce_loss += ce       # append total CE loss across all classes
            print(ce_loss)
#        ce_loss = torch.sum(ce_list)  # total CE loss across all classes
        
        return ce_loss


# utils.py
functions necessary for splitting data, plotting learning curves etc

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Script for visualising training results. 
Read .txt files of train and val acc/loss and plots over epochs.

Created on Fri Nov 13 13:26:48 2020

@author: calmac
"""
import numpy as np
import matplotlib.pyplot as plt
import os

def plotLearningCurves(log_root, save=False):   
    train_stats_dir = os.path.join(log_root,'train.txt')
    val_stats_dir = os.path.join(log_root, 'val.txt')
    with open(train_stats_dir) as f:
        train_dice, train_loss = [], []
        i=0
        for line in f.readlines():
          if i%2 == 0:
              train_dice.append(np.array(line).astype(np.float32))  # even rows are dice
          else:
              train_loss.append(np.array(line).astype(np.float32))  # odd rows are loss
          i+=1
        f.close()
    with open(val_stats_dir) as f:
        val_dice, val_loss = [], []
        i=0
        for line in f.readlines():
          if i%2 == 0:
              val_dice.append(np.array(line).astype(np.float32))
          else:
              val_loss.append(np.array(line).astype(np.float32))
          i+=1
        f.close()  
    # Plot dice
    fig = plt.figure()
    plt.plot(train_dice,'b', val_dice, 'r')
    plt.xlabel('Epoch')
    plt.ylabel('Dice')
    plt.legend(('training','validation'))
    plt.show
    plt.savefig(os.path.join(log_root,'diceCurve.png'))

    # Plot loss
    fig = plt.figure()
    plt.plot(train_loss,'b--', val_loss, 'r--')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(('training','validation'))
    plt.show
    plt.savefig(os.path.join(log_root,'lossCurve.png'))

#    return {'train_stats':[train_dice,train_loss], 'val_stats':[val_dice,val_loss]}


    
    

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Thu Nov 12 20:25:56 2020

@author: hsijcr
"""

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import numpy as np
import os
import glob

def assign_val_data():
  file_list = glob.glob(root_trainData)
  train_val_dict = split_train_val(file_list, val_percent)
  return train_val_dict

def split_train_val(dataset, val_percent):
  length = len(dataset)
  n = int(length * val_percent)
  return {'train': dataset[:-n], 'val': dataset[-n:]} # train = all data minus number of validation examples
                                                        # val   = the remaining number of examples

  

actual utils.py from PC

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import numpy as np
import torch
import os
import os.path as osp
import cv2
import scipy.misc as misc
import shutil
from skimage import measure
import math
import traceback
from sklearn import metrics
import zipfile
from torch.nn import init

### initalize the module
def init_weights(net, init_type='normal'):
    #print('initialization method [%s]' % init_type)
    if init_type == 'kaiming':
        net.apply(weights_init_kaiming)
    else:
        raise NotImplementedError('initialization method [%s] is not implemented' % init_type)

def weights_init_kaiming(m):
    classname = m.__class__.__name__
    #print(classname)
    if classname.find('Conv') != -1:
        init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
    elif classname.find('Linear') != -1:
        init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
    elif classname.find('BatchNorm') != -1:
        init.normal_(m.weight.data, 1.0, 0.02)
        init.constant_(m.bias.data, 0.0)

### compute model params
def count_param(model):
    param_count = 0
    for param in model.parameters():
        param_count += param.view(-1).size()[0]
    return param_count

class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
        
def save_checkpoint(state, is_best,checkpoint_path,filename='./checkpoint/checkpoint.pth.tar'):
    torch.save(state, filename)
    if is_best:
        shutil.copyfile(filename, osp.join(checkpoint_path,'model_best.pth.tar'))

def save_dice_single(is_best, filename='dice_single.txt'):
    if is_best:
        shutil.copyfile(filename, 'dice_best.txt')
        
def compute_dice_score(predict, gt, forground = 1):
    score = 0
    count = 0
    assert(predict.shape == gt.shape)
    overlap = 2.0 * ((predict == forground)*(gt == forground)).sum()
    #print('overlap:',overlap)
    
    return (overlap + 0.001) / (((predict == forground).sum() + (gt == forground).sum()) + 0.001)

def compute_average_dice(predict, gt, class_num = 4):
    Dice = 0
    Dice_list = []

    for i in range(1,class_num):
        predict_copy = predict.copy()
        gt_copy = gt.copy()
        predict_copy[predict_copy != i] = 0
        gt_copy[gt_copy != i] = 0
        dice = compute_dice_score(predict_copy, gt_copy, forground = i)
        Dice += dice
        Dice_list.append(dice)
    return Dice/(class_num - 1),Dice_list[0],Dice_list[1],Dice_list[2]

def compute_score(predict, gt, forground = 1):
    score = 0
    count = 0
    assert(predict.shape == gt.shape)
    overlap = ((predict == forground)*(gt == forground)).sum()
    #print('overlap:',overlap)
    if(overlap > 0):
        return 2*overlap / ((predict == forground).sum() + (gt == forground).sum()),overlap /  (predict == forground).sum(), overlap / (gt == forground).sum(),overlap / ((predict == forground).sum() + (gt == forground).sum() - overlap)
        # dice,precsion,recall
    else:
        return 0,0,0,0
    
def eval_seg(predict, gt, forground = 1):
    assert(predict.shape == gt.shape)
    Dice = 0
    Precsion = 0
    Recall = 0
    Jaccard = 0
    n = predict.shape[0]
    for i in range(n):
        dice,precsion,recall,jaccard = compute_score(predict[i],gt[i])
        Dice += dice
        Precsion += precsion
        Recall += recall
        Jaccard += jaccard

    return Dice/n,Precsion/n,Recall/n,Jaccard/n

def aic_fundus_lesion_classification(ground_truth, prediction, num_samples=128):
    """
    Classification task auc metrics.
    :param ground_truth: numpy matrix, (num_samples, 3)
    :param prediction: numpy matrix, (num_samples, 3)
    :param num_samples: int, default 128
    :return list:[AUC_1, AUC_2, AUC_3]
    """
    # assert (ground_truth.shape == (num_samples, 3))
    # assert (prediction.shape == (num_samples, 3))

    try:
        ret = [0.5, 0.5, 0.5]
        for i in range(3):
            fpr, tpr, thresholds = metrics.roc_curve(ground_truth[:, i], prediction[:, i], pos_label=1)
            ret[i] = metrics.auc(fpr, tpr)

        
        # fpr, tpr, thresholds = metrics.roc_curve(ground_truth[:,0], prediction[:,0], pos_label=1)
        # ret = metrics.auc(fpr, tpr)
    except Exception as e:
        traceback.print_exc()
        print("ERROR msg:", e)
        return None
    return ret

def aic_fundus_lesion_segmentation(ground_truth, prediction, num_samples=128):
    """
    Detection task auc metrics.
    :param ground_truth: numpy matrix, (num_samples, 1024, 512)
    :param prediction: numpy matrix, (num_samples, 1024, 512)
    :param num_samples: int, default 128
    :return list:[Dice_0, Dice_1, Dice_2, Dice_3]
    """
    #assert (ground_truth.shape == (num_samples, 1024, 512))
    #assert (prediction.shape == (num_samples, 1024, 512))

    ground_truth = ground_truth.flatten()
    prediction = prediction.flatten()
    try:
        ret = [0.0, 0.0, 0.0, 0.0]
        for i in range(4):
            mask1 = (ground_truth == i)
            mask2 = (prediction == i)
            if mask1.sum() != 0:
                ret[i] = 2 * ((mask1 * (ground_truth == prediction)).sum()) / (mask1.sum() + mask2.sum())
            else:
                ret[i] = float('nan')
    except Exception as e:
        traceback.print_exc()
        print("ERROR msg:", e)
        return None
    return ret

def compute_segment_score(ret_segmentation,cubes=15):
    REA_segementation, SRF_segementation, PED_segementation = 0.0, 0.0, 0.0
    n1, n2, n3 = 0, 0, 0
    for i in range(cubes):
        if not math.isnan(ret_segmentation[i][1]):
            REA_segementation += ret_segmentation[i][1]
            n1 += 1
        if not math.isnan(ret_segmentation[i][2]):
            SRF_segementation += ret_segmentation[i][2]
            n2 += 1
        if not math.isnan(ret_segmentation[i][3]):
            PED_segementation += ret_segmentation[i][3]
            n3 += 1

    REA_segementation /= n1
    SRF_segementation /= n2
    PED_segementation /= n3
    avg_segmentation = (REA_segementation + SRF_segementation + PED_segementation) / 3

    return avg_segmentation,REA_segementation,SRF_segementation,PED_segementation

def compute_single_segment_score(ret_segmentation):
    REA_segementation, SRF_segementation, PED_segementation = 0.0, 0.0, 0.0
    n1, n2, n3 = 0, 0, 0
    
    if not math.isnan(ret_segmentation[1]):
        REA_segementation += ret_segmentation[1]
        n1 += 1
    if not math.isnan(ret_segmentation[2]):
        SRF_segementation += ret_segmentation[2]
        n2 += 1
    if not math.isnan(ret_segmentation[3]):
        PED_segementation += ret_segmentation[3]
        n3 += 1

    avg_segmentation = (REA_segementation + SRF_segementation + PED_segementation) / (n1+n2+n3)

    return avg_segmentation

def rebuild_tensor_v2():
    try:
        torch._utils._rebuild_tensor_v2
    except AttributeError:
        def _rebuild_tensor_v2(storage, storage_offset, size, stride, requires_grad, backward_hooks):
            tensor = torch._utils._rebuild_tensor(storage, storage_offset, size, stride)
            tensor.requires_grad = requires_grad
            tensor._backward_hooks = backward_hooks
            return tensor
        torch._utils._rebuild_tensor_v2 = _rebuild_tensor_v2
        
def target_seg2target_cls(array):
    class_num_all = torch.zeros(array.shape[0],3)
    for i in range(array.shape[0]):
        array_np = np.unique(array[i])
        label = array_np[1:] + 1
        if label.sum() == 0:
            class_num = torch.tensor([0,0,0])
        elif label.sum() == 2:#np.array([0,255]):
            class_num = torch.tensor([1,0,0])
        elif label.sum() == 3:#np.array([0,191]):
            class_num = torch.tensor([0,1,0])
        elif label.sum() == 4:#np.array([0,128]):
            class_num = torch.tensor([0,0,1])
        elif label.sum() == 5: #np.array([0,255,191]):
            class_num = torch.tensor([1,1,0])
        elif label.sum() == 6:#np.array([0,255,128]):
            class_num = torch.tensor([1,0,1])
        elif label.sum() == 7:#np.array([0,191,128]):
            class_num = torch.tensor([0,1,1])
        elif label.sum() == 9:#np.array([0,255,191,128]):
            class_num = torch.tensor([1,1,1])
        class_num_all[i,:] = class_num

    return class_num_all.float()

def zip_dir(dirname,zipfilename):
    filelist = []
    if osp.isfile(dirname):
        filelist.append(dirname)
    else :
        for root, dirs, files in os.walk(dirname):
            for name in files:
                filelist.append(osp.join(root, name))
        
    zf = zipfile.ZipFile(zipfilename, "w", zipfile.zlib.DEFLATED)
    for tar in filelist:
        arcname = tar[len(dirname):]
        #print arcname
        zf.write(tar,arcname)
    zf.close()

# model.py
contains the Unet code and associated functions for building it.

unet_parts.py

In [8]:
# sub-parts of the U-Net model

import torch
import torch.nn as nn
import torch.nn.functional as F


class double_conv(nn.Module):
    '''(conv => BN => ReLU) * 2'''
    def __init__(self, in_ch, out_ch):
        super(double_conv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            # nn.GroupNorm(32, out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            # nn.GroupNorm(32, out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.conv(x)
        return x
    
class single_conv(nn.Module):
    '''(conv => BN => ReLU) * 1'''
    def __init__(self, in_ch, out_ch):
        super(single_conv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            # nn.GroupNorm(32, out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.conv(x)
        return x


class inconv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(inconv, self).__init__()
        self.conv = double_conv(in_ch, out_ch)

    def forward(self, x):
        x = self.conv(x)
        return x


class down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(down, self).__init__()
        self.mpconv = nn.Sequential(
            nn.MaxPool2d(2),
            double_conv(in_ch, out_ch)
        )

    def forward(self, x):
        x = self.mpconv(x)
        return x


class up(nn.Module):
    def __init__(self, in_ch, out_ch, bilinear=True):
        super(up, self).__init__()

        #  would be a nice idea if the upsampling could be learned too,
        #  but my machine do not have enough memory to handle all those weights
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose2d(in_ch//2, in_ch//2, 2, stride=2)

        self.conv = double_conv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, (diffX // 2, diffX - diffX//2,
                        diffY // 2, diffY - diffY//2))
        
        # for padding issues, see 
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd

        x = torch.cat([x2, x1], dim=1)
        x = self.conv(x)
        return x


class outconv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(outconv, self).__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x):
        x = self.conv(x)
        return x

class edgeconv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(edgeconv, self).__init__()
        self.conv = double_conv(in_ch, out_ch)

    def forward(self, x):
        x = self.conv(x)
        return x
    
class fuse(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(fuse, self).__init__()
        self.conv = single_conv(in_ch, out_ch)

    def forward(self, x):
        x = self.conv(x)
        return x       
        

unet_vanilla.py

In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Fri Sep 20 11:29:46 2019

@author: hsijcr
adapted by me 
"""

import torch.nn.functional as F
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes):
        super(UNet, self).__init__()
        self.inc = inconv(n_channels, 64)
        self.down1 = down(64, 128)
        self.down2 = down(128, 256)
        self.down3 = down(256, 512)
        self.down4 = down(512, 512)
#        self.down5 = down(1024,1024)
#        self.up5 = up(2048, 512)
        self.up4 = up(1024, 256)
        self.up3 = up(512, 128)
        self.up2 = up(256, 64)
        self.up1 = up(128,64)
        self.outc = outconv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
#        x6 = self.down5(x5)
        
#        x = self.up5(x6, x5)
        x = self.up4(x5, x4)
        x = self.up3(x, x3)
        x = self.up2(x, x2)
        x = self.up1(x, x1)
#        x = self.up1(x5, x4)
#        x = self.up2(x, x3)
#        x = self.up3(x, x2)
#        x = self.up4(x, x1)
        x = self.outc(x)
        return F.log_softmax(x, dim=1)
    
def model_param_count(model):
    return sum(p.numel() for p in model.parameters())
        


unet_parts_bn.py

# train.py
everything in the train.py code, including train() and validate() steps.

In [10]:
#!/usr/bin/env python3
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Thu 12th November 

Script for performing both training and validation steps on the cardiac MRI data.

@author: calmac

"""
# Normal python stuff
from datetime import datetime
import os
import os.path
import numpy as np
# Pytorch 
import torch
import torch.autograd.variable as Variable
import torch.optim as optim
import torch.utils.data
import torch.nn.functional as F


def main(dataloaders, classifier, optimizer, scheduler, dice_loss):

  print('Training model...')
  for epoch in range(num_epochs):
    print('EPOCH %03d ' % (epoch+1))
    
    train_dataloader = dataloaders['train']
    val_dataloader = dataloaders['val']

    # Run a training step
    train_step(classifier, dice_loss, train_dataloader, scheduler, optimizer, epoch)
    
    # Run a validation step
    with torch.no_grad():
        validation_step(classifier, dice_loss, val_dataloader, epoch)
  
  # Display learning curve and update each epoch
  plotLearningCurves(log_root, save=True)
    
def train_step(classifier, dice_loss, dataloader, scheduler, optimizer, epoch):
  
  classifier.train() # switch model to training mode
  
  Dice = AverageMeter()
  Dice_rv  = AverageMeter()
  Dice_myo = AverageMeter()
  Dice_lv  = AverageMeter()
  train_loss_epoch, train_dice_epoch = [], []
#  gradients = []
  for i, (slices,label) in enumerate(dataloader):

    slices, label = slices.to(device), label.to(device)
#    slices = torch.tensor(slices, requires_grad=True) # need this for tracking gradients
    optimizer.zero_grad()
    pred = classifier(slices)
    # Compute losses and backpropagate gradients
    loss1 = dice_loss(pred, label.long())
    pred = pred.permute(0,2,3,1).contiguous().view(-1, n_classes)
    label = label.view(-1).long()
    loss2 = F.nll_loss(pred, label)
    loss = loss1+loss2
    loss.backward()
    
    # Save gradients and update parameters
#    print(slices.grad.size())
#    grad = torch.mean(slices.grad.clone())
#    gradients.append(grad.detach().cpu().numpy())
    optimizer.step()
#    print(l)
    
    # Compute performance
    pred_seg = pred.data.max(1)[1].cpu().numpy()
    label_seg = label.data.cpu().numpy()
    dice_score, dice1, dice2, dice3 = compute_average_dice(pred_seg, label_seg)
    Dice.update(dice_score)
    Dice_rv.update(dice1)
    Dice_myo.update(dice2)
    Dice_lv.update(dice3)
    train_loss_epoch.append(loss.detach().cpu().numpy())
    train_dice_epoch.append(Dice)
  scheduler.step()
  print('Training stats:')
  print(('\tRV Dice:   %f') % (Dice_rv.avg))
  print(('\tMyo Dice:  %f') % (Dice_myo.avg))
  print(('\tLV Dice:   %f') % (Dice_lv.avg))
  print(('\tMean Dice: %f') % (Dice.avg))
  print(('\tLoss: %f') % (np.mean(train_loss_epoch)))
#  print('\tMean grads over epoch={}'.format(np.mean(gradients)))
  
  # Semd to files
  train_log_string('**** EPOCH %03d ****' % (epoch+1))
  train_log_string(str(datetime.now()))
  train_log_string(('epoch %d | train dice: %f') % (epoch+1, Dice.avg))
  train_log_string(('epoch %d | train dice 1: %f') % (epoch+1, Dice_rv.avg))
  train_log_string(('epoch %d | train dice 2: %f') % (epoch+1, Dice_myo.avg))
  train_log_string(('epoch %d | train dice 3: %f') % (epoch+1, Dice_lv.avg))
  train_log_string(('epoch %d | mean train loss: %f') % (epoch+1, np.mean(train_loss_epoch)))
  train_txt_string(('%f') % (Dice.avg))
  train_txt_string(('%f') % (np.mean(train_loss_epoch)))
#  grad_string(('%f') % (np.mean(gradients)))
  if save_model and (epoch+1) % log_every == 0:
      torch.save(classifier.state_dict(), '%s/%s_model_%d.pth' % (weights_path, 'acdc', epoch+1))
       
def validation_step(classifier, dice_loss, dataloader, epoch):
  
  classifier.eval() # switch to evaluation mode
  
  Dice = AverageMeter()
  Dice_rv  = AverageMeter()
  Dice_myo = AverageMeter()
  Dice_lv  = AverageMeter()
  val_loss_epoch, val_dice_epoch = [], []
  
  for i, (slices,label) in enumerate(dataloader):

    slices, label = slices.to(device), label.to(device)
    pred = classifier(slices)
    
    # Compute losses
    loss1 = dice_loss(pred, label.long())
    pred = pred.permute(0,2,3,1).contiguous().view(-1, n_classes)
    label = label.view(-1).long()
    loss2 = F.nll_loss(pred, label)
    loss = loss1+loss2
    
    # Compute performance
    pred_seg = pred.data.max(1)[1].cpu().numpy()
    label_seg = label.data.cpu().numpy()
    dice_score, dice1, dice2, dice3 = compute_average_dice(pred_seg, label_seg)
    Dice.update(dice_score)
    Dice_rv.update(dice1)
    Dice_myo.update(dice2)
    Dice_lv.update(dice3)
    val_loss_epoch.append(loss.detach().cpu().numpy())
    val_dice_epoch.append(Dice)
  print('Validation stats:')
  print(('\tRV Dice:   %f') % (Dice_rv.avg))
  print(('\tMyo Dice:  %f') % (Dice_myo.avg))
  print(('\tLV Dice:   %f') % (Dice_lv.avg))
  print(('\tMean Dice: %f') % (Dice.avg))
  print(('\tLoss: %f') % (np.mean(val_loss_epoch)))
  val_log_string('**** EPOCH %03d ****' % (epoch+1))
  val_log_string(str(datetime.now()))
  val_log_string(('epoch %d | val dice: %f') % (epoch+1, Dice.avg))
  val_log_string(('epoch %d | val dice 1: %f') % (epoch+1, Dice_rv.avg))
  val_log_string(('epoch %d | val dice 2: %f') % (epoch+1, Dice_myo.avg))
  val_log_string(('epoch %d | val dice 3: %f') % (epoch+1, Dice_lv.avg))
  val_log_string(('epoch %d | mean train loss: %f') % (epoch+1, np.mean(val_loss_epoch)))
  val_txt_string(('%f') % (Dice.avg))
  val_txt_string(('%f') % (np.mean(val_loss_epoch)))



# Train model


Initialise dataloaders, model, and param update protocols 

In [11]:
# Set up folders for storing stuff
log_root = stats_root            # folder for storing train/val progress 
if not os.path.exists(log_root): os.mkdir(log_root)
trainResultsFile_all = open(os.path.join(log_root, 'train.log'), 'w')
valResultsFile_all = open(os.path.join(log_root, 'val.log'), 'w')
trainResultsFile = open(os.path.join(log_root, 'train.txt'), 'w')
valResultsFile = open(os.path.join(log_root, 'val.txt'), 'w')
#gradFile = open(os.path.join(log_root, 'grad.txt'), 'w')

def train_log_string(out_str):
  trainResultsFile_all.write(out_str+'\n')
  trainResultsFile_all.flush()
def train_txt_string(out_str):
  trainResultsFile.write(out_str+'\n')
  trainResultsFile.flush()
def val_log_string(out_str):
  valResultsFile_all.write(out_str+'\n')
  valResultsFile_all.flush()
def val_txt_string(out_str):
  valResultsFile.write(out_str+'\n')
  valResultsFile.flush()
#def grad_string(out_str):
#  gradFile.write(out_str+'\n')
#  gradFile.flush()  
  
# Establish available device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Setup datasets  
train_val_dict = assign_val_data()
train_list = train_val_dict['train']
val_list = train_val_dict['val']

# Assign datasets and dataloaders
train_dataset = dataset(train_list, train=True, transform=transform)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_dataset =  dataset(val_list, train=True, transform=transform)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, num_workers=num_workers)
print('Training dataset size: {}'.format(len(train_dataset)))
print('Validation dataset size: {}'.format(len(val_dataset)))
dataloaders = {'train':train_dataloader, 'val':val_dataloader}

# Build model.
classifier = UNet(n_channels=in_channels, n_classes=n_classes)
classifier.to(device)
print('{} built.'.format(model_type))
param_count = model_param_count(classifier)
print('Model contains {:4.4f} M parameters.'.format(param_count / 1e6))
print('learning rate= {}'.format(lr))
print('batch size= {}'.format(batch_size))

# Assign optimiser and cost function
optimizer = optim.Adam(classifier.parameters(), lr=lr, betas=(beta1,beta2), weight_decay=weight_decay)
# optimizer = optim.SGD(classifier.parameters(), lr=lr, momentum=momentum, nesterov=False)
#  scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=T_0, T_mult=T_mult, eta_max=lr, T_up=T_up, gamma=gamma)
#  scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=gamma)
dice_loss = DiceLoss()


Training dataset size: 1359
Validation dataset size: 151
U-Net built.
Model contains 13.3955 M parameters.
learning rate= 0.001
batch size= 200


Call main() and start training

In [12]:
main(dataloaders, classifier, optimizer, scheduler, dice_loss)

Training model...
EPOCH 001 


RuntimeError: ignored